# Tutorial 2: Person Re-Identification (ReID) Across Cameras

**Pipeline Stage:** Matching the same person across different camera views

---

## Overview

This tutorial covers the **second stage** of the multi-camera human tracking pipeline:
using **appearance-based embeddings** to determine which person detected in one camera
is the same person seen in another camera.

### Why ReID BEFORE Triangulation?

This is the critical insight: **triangulation requires knowing which detections across
cameras correspond to the same person.** If Camera 1 sees 5 people and Camera 3 sees
5 people, triangulation needs to know that "Person 2 in Cam1" = "Person 4 in Cam3"
before it can combine their 2D keypoints into 3D coordinates.

Without ReID, you'd be triangulating random pairings of different people — garbage in,
garbage out.

### What You Will Learn

1. **Installation** — BoxMOT, torchreid, OSNet weights, and dependencies
2. **Torso Cropping** — Extracting shirt/torso crops from video + pose `.slp`, without distorting them
3. **ReID Embeddings** — Turning each crop into a 512-dim appearance vector with OSNet
4. **Cosine Similarity** — Measuring appearance similarity, and why you must split pairs by type
5. **Identity Assignment** — Hungarian matching + temporal constraints to turn embeddings into decisions
6. **Writing Tracks with `sleap-io`** — Exporting `.slp` files where each person is a `Track`
7. **Proofreading in SLEAP** — Reviewing and correcting the automated identities
8. **The Identity Map** — Cross-camera correspondences for triangulation

### The Key Output

By the end you will have **one tracked `.slp` file per camera** in `reid_results/` —
`reid_<CAM>.slp` — openable in the SLEAP GUI, where the track named `person_2` refers to the
same human in *every* file. That is both the input to triangulation and the thing you
proofread.

### Pipeline Context

```
┌─────────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│  Tutorial 1          │ ──► │  Tutorial 2 (HERE)    │ ──► │  Tutorial 3          │
│  YOLO Pose (2D)     │     │  Person ReID          │     │  3D Triangulation    │
│  per-camera          │     │  cross-camera match   │     │  multi-camera fusion │
└─────────────────────┘     └──────────────────────┘     └─────────────────────┘
                                      │
                                      ▼
                         For each frame, answers:
                         "Detection 2 in CAM1 =
                          Detection 0 in CAM3 =
                          Detection 1 in CAM5"
```

---

## Before You Start

### You need four things

| # | What | Where it comes from |
|---|---|---|
| 1 | **Synchronized videos**, one per camera | your recording setup |
| 2 | **A pose `.slp` per video** | **Tutorial 1** — untracked detections are fine, that is exactly what we start from |
| 3 | **OSNet ReID weights** — `osnet_x0_25_imagenet.pth` | [deep-person-reid](https://github.com/KaiyangZhou/deep-person-reid) |
| 4 | **How many people** are in your scene | you — this is the `N_PEOPLE` setting |

> **"Synchronized" matters.** Cross-camera matching compares detections *at the same frame
> index* in different videos. If your videos are not frame-aligned, frame 500 in CAM1 and
> frame 500 in CAM2 show different moments and cross-camera ReID cannot work. Align them
> first.

### Where to put your data

You do not have to move anything — **you point the notebook at your files** in STEP 3. The
only assumption is that each pose file is named after its video:

```
<video_stem>.mp4        ->        <video_stem>_pose.slp
```

which is exactly what Tutorial 1 produces. A typical layout:

```
my_experiment/
├── videos/
│   ├── CAM1_session.mp4          <- VIDEO_DIR
│   ├── CAM2_session.mp4
│   └── ...
├── videos/pose_results/          <- POSE_DIR   (Tutorial 1 output)
│   ├── CAM1_session_pose.slp
│   ├── CAM2_session_pose.slp
│   └── ...
└── models/
    └── osnet_x0_25_imagenet.pth  <- REID_WEIGHTS
```

Videos and pose files may live in the same directory, or the pose files may sit in
per-camera subdirectories — STEP 3 handles both. Nothing is ever written back into your
data directories; all output goes to `reid_results/` next to this notebook.

### Hardware

A GPU is recommended. One embedding is computed per detection, so a multi-camera clip
easily reaches 10^5 crops — minutes on a GPU, hours on a CPU. To try the notebook quickly
on CPU, raise `FRAME_STRIDE` in STEP 3 (e.g. `10`) and everything still runs, just on fewer
frames.

> **Note on file formats:** this notebook reads pose `.slp` files directly rather than SLEAP
> `.analysis.h5` exports. The `.h5` format requires a fixed instance count for the whole video
> and only exists *after* tracking has already assigned identities — neither is true of our
> input.

---

## Part 1: Installation

| Package | Purpose | Install |
|---|---|---|
| `boxmot` | ReID encoder (OSNet) wrapper | `pip install boxmot` |
| `torch` / `torchvision` | Deep learning framework | `pip install torch torchvision` |
| `sleap-io` | Reading and writing `.slp` files | `pip install sleap-io` |
| `scipy` | Hungarian algorithm (`linear_sum_assignment`) | `pip install scipy` |
| `opencv-python` | Video reading, image cropping | `pip install opencv-python` |
| `matplotlib` | Visualization | `pip install matplotlib` |
| `tqdm` | Progress bars | `pip install tqdm` |

### ReID Model: OSNet

We use **OSNet (Omni-Scale Network)** — a lightweight model designed specifically for
person re-identification. The `osnet_x0_25` variant is:
- Very fast (~1 ms per crop on GPU)
- Produces 512-dimensional appearance embeddings
- Pre-trained on ImageNet, fine-tunable on ReID datasets

### ReID Weights

Download pre-trained weights:
- `osnet_x0_25_imagenet.pth` — general-purpose, works well out of the box
- Available from the [deep-person-reid](https://github.com/KaiyangZhou/deep-person-reid) repository

> **On `boxmot` versions:** BoxMOT has moved its internal ReID encoder between attributes
> across releases (`.encoder`, `.model`, and a separate `ReidAutoBackend`). STEP 5 probes for a
> working encoder and validates it with a dummy forward pass instead of assuming one layout, so
> it should survive a version bump. If all paths fail it falls back to `torchreid` directly.


In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# Uncomment and run if not yet installed:

# !pip install boxmot torch torchvision sleap-io scipy opencv-python matplotlib tqdm

# sleap-io is the one to check if you hit trouble writing .slp files — this notebook
# is tested against 0.2.x and 0.3.x and handles the API differences between them.


In [ ]:
# ============================================================
# STEP 2: Verify installation and GPU
# ============================================================
import sys

print(f"Python:     {sys.version.split()[0]}")

import torch
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("MPS:        available (Apple Silicon)")
else:
    print("            no GPU — embedding will be slow, raise FRAME_STRIDE in STEP 3")

import cv2
print(f"OpenCV:     {cv2.__version__}")

import numpy as np
print(f"NumPy:      {np.__version__}")

import sleap_io as sio
print(f"sleap-io:   {sio.__version__}")

import scipy
from scipy.optimize import linear_sum_assignment
print(f"SciPy:      {scipy.__version__}  (linear_sum_assignment OK)")

import matplotlib
print(f"matplotlib: {matplotlib.__version__}")

try:
    import boxmot
    print(f"BoxMOT:     {getattr(boxmot, '__version__', 'installed')}")
except ImportError:
    try:
        import torchreid
        print("BoxMOT:     NOT installed — will fall back to torchreid in STEP 5")
    except ImportError:
        print("BoxMOT:     NOT installed, and no torchreid fallback.")
        print("            Install with: pip install boxmot")


---

## Part 2: Understanding the ReID Pipeline

### The Problem

Tutorial 1 gave us, for every camera and every frame, a set of skeletons — but with **no
identities**. Two questions are unanswered:

```
WITHIN a camera, across time:
  Is the person at the left of frame 100 the same as the one at the left of frame 101?
  -> without this, nobody has a trajectory

ACROSS cameras, at one instant:
  Is CAM1's second detection the same human as CAM3's fourth?
  -> without this, triangulation combines different people's body parts
```

Both must be answered before triangulation. If CAM1 sees 6 people and CAM3 sees 6 people,
there are 6! = 720 possible pairings and only one is right.

### The Approach

```
Video + pose .slp (all cameras)
       |
       v
+---------------+   +----------------+   +------------------+   +---------------+
|  Crop torsos  |-->|  OSNet         |-->|  Assign          |-->|  Write .slp   |
|  from frames  |   |  embeddings    |   |  identities      |   |  with Tracks  |
+---------------+   +----------------+   +------------------+   +---------------+
  shoulders-hips      512-dim vector       Hungarian +             sleap-io
  aspect preserved    per detection        time constraints        -> PROOFREAD
```

The third box is where this notebook does its real work, and it is the part a similarity
matrix alone cannot do — see Part 7.

### Why Torso Crops?

In classrooms and camps the lower body is occluded by desks, chairs and other people. Shirt
colour and pattern are usually the most discriminative signal available, so a torso crop
often **outperforms** a full-body crop — the full-body box is mostly furniture, and furniture
looks identical for everybody.

### Why the Output Goes to a Human

Appearance-based ReID on six children, several in matching camp T-shirts, will make mistakes.
Rather than hide that, we write the result to `.slp` and review it in the SLEAP GUI. Fixing an
identity swap at the frame where it happens is one click; discovering it after triangulation
means debugging a corrupted 3D trajectory.


---

## Part 3: Point the Notebook at Your Data

**This is the only cell you need to edit.** STEP 3 below has four paths at the top; set them
and run it. It then finds your cameras automatically and tells you exactly what it found.

### The four settings

| Setting | What to put there |
|---|---|
| `VIDEO_DIR` | folder containing your camera videos |
| `POSE_DIR` | folder containing Tutorial 1's `*_pose.slp` files |
| `REID_WEIGHTS` | path to `osnet_x0_25_imagenet.pth` |
| `N_PEOPLE` | how many distinct people to resolve |

Everything else has a working default.

### How cameras are discovered

STEP 3 globs `VIDEO_DIR` for videos, and for each one looks for a matching pose file:

```
VIDEO_DIR/CAM1_session.mp4   ──►   POSE_DIR/CAM1_session_pose.slp
             │                                    │
             └── video stem ─────────────────────┘  + POSE_SUFFIX
```

Videos without a matching pose file are **skipped and reported**, so extra files in the
folder (proxies, `_30fps` copies, unrelated recordings) do not break anything.

The short camera label comes from `CAMERA_REGEX` applied to the filename:

| Your filenames | Set `CAMERA_REGEX` to | Labels you get |
|---|---|---|
| `CAM1_session.mp4` | `r"(CAM\d+)"` *(default)* | `CAM1`, `CAM2`, … |
| `recording_cam02.mp4` | `r"(cam\d+)"` | `cam02`, … |
| `left.mp4`, `right.mp4` | `None` | `left`, `right` |

If two videos produce the same label, the second is skipped with a warning — tighten the
regex so each camera gets a unique name.

### Common layout variations

- **Poses next to videos?** Set `POSE_DIR = VIDEO_DIR`.
- **Poses in per-camera subfolders?** Leave `POSE_DIR` as the parent — STEP 3 searches
  recursively for the matching filename.
- **Videos in per-camera subfolders?** Set `VIDEO_GLOB = "**/*.mp4"`.
- **Not `.mp4`?** Set `VIDEO_GLOB = "*.mkv"` (or `"*.avi"`, …).
- **Only one camera?** Stages 1–2 (tracking within a camera) still work and still give you a
  tracked `.slp`. Stage 3 needs at least two.

---

### Technical note: why `.slp` and not `.h5`

Tutorial 1's `.slp` files contain **untracked detections** — every frame has some number of
`Instance` objects with keypoints, but **no identity**. Nothing links "the person on the left
in frame 100" to "the person on the left in frame 101". That is the gap this tutorial closes.

We read them with `sleap-io`:

```python
labels = sio.load_slp(path)
for lf in labels:                    # each LabeledFrame
    for inst in lf.instances:        # variable count per frame
        pts = inst.numpy()           # (n_nodes, 2), NaN where not visible
```

This handles a varying number of people per frame, which is what real detector output looks
like — and what the dense `.h5` array format cannot represent.

### Technical note: the join key

Everything downstream is keyed on the triple:

```
(camera, frame_idx, instance_idx)
```

`instance_idx` is the position of the instance within its `LabeledFrame`. We carry this key
alongside every crop and every embedding so that at the very end we can walk back through the
original `.slp` and attach the computed identity to exactly the right instance.


In [ ]:
# ============================================================
# STEP 3: Configuration  <<<<<  EDIT THE PATHS IN THIS CELL
# ============================================================
import json
import os
import re
from pathlib import Path

import cv2
import numpy as np
import sleap_io as sio
from tqdm import tqdm

# =============================================================
#  1. YOUR DATA  <<<<< change these four lines
# =============================================================
VIDEO_DIR = Path("PUT_YOUR_VIDEO_FOLDER_HERE")
POSE_DIR = Path("PUT_YOUR_TUTORIAL_1_POSE_FOLDER_HERE")
REID_WEIGHTS = Path("PUT_PATH_TO_osnet_x0_25_imagenet.pth_HERE")

N_PEOPLE = 6        # how many distinct people to resolve. THE most important setting.

# A filled-in example, for reference:
#   VIDEO_DIR    = Path("/data/my_experiment/videos")
#   POSE_DIR     = Path("/data/my_experiment/videos/pose_results")
#   REID_WEIGHTS = Path("/data/models/osnet_x0_25_imagenet.pth")
#   N_PEOPLE     = 4
#
# Windows users: use forward slashes or a raw string --
#   VIDEO_DIR = Path("D:/my_experiment/videos")     # fine
#   VIDEO_DIR = Path(r"D:\my_experiment\videos")    # also fine

OUT_DIR = Path("reid_results")     # all output lands here, next to this notebook

# =============================================================
#  2. FILE NAMING  (defaults match Tutorial 1's output)
# =============================================================
VIDEO_GLOB = "*.mp4"          # "*.mkv", or "**/*.mp4" for per-camera subfolders
POSE_SUFFIX = "_pose.slp"     # Tutorial 1 writes "<video_stem>_pose.slp"

# How to derive a short camera label from a filename; first regex group wins.
# Use None to fall back to the whole video stem.
#   "CAM1_session.mp4"     + r"(CAM\d+)"  -> "CAM1"
#   "recording_cam02.mp4"  + r"(cam\d+)"  -> "cam02"
#   "left.mp4"             + None         -> "left"
# If the regex simply does not match a filename, the whole video stem is used instead --
# so leaving the default in place is harmless for non-CAM naming schemes.
CAMERA_REGEX = r"(CAM\d+)"

# =============================================================
#  3. SPEED
# =============================================================
FRAME_STRIDE = 1      # 1 = every frame. Try 10 for a quick look, or on CPU.

# =============================================================
#  4. ALGORITHM PARAMETERS  (sensible defaults -- tune later)
# =============================================================
CROP_W, CROP_H = 128, 256   # OSNet input size (W, H) -- person-ReID standard 1:2 aspect
TORSO_PAD = 0.15            # pad the torso box by this fraction of its size
BATCH_SIZE = 256            # crops per forward pass; lower it if you run out of VRAM

# Within-camera tracklet linking (STEP 10)
W_APPEARANCE = 0.6          # weight on (1 - cosine similarity)
W_MOTION = 0.4              # weight on normalized centroid movement
MAX_MOVE_FRAC = 0.15        # max jump per frame, as a fraction of the frame diagonal
MAX_LINK_COST = 0.7         # above this, start a new tracklet instead of linking
MAX_AGE = 30                # retire a tracklet after this many frames unseen
MIN_TRACKLET_LEN = 5        # ignore tracklets shorter than this when consolidating

# Skeleton layout -- COCO/YOLO defaults from Tutorial 1. TORSO_IDX only needs to point at
# shoulders and hips; change both if your skeleton differs.
NODE_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
TORSO_IDX = [5, 6, 11, 12]   # left/right shoulder, left/right hip

# =============================================================
#  Nothing below here needs editing.
# =============================================================
def camera_name(video_path):
    """Short label for a camera, derived from its video filename."""
    if CAMERA_REGEX:
        m = re.search(CAMERA_REGEX, video_path.name)
        if m:
            return m.group(1)
    return video_path.stem


def find_pose_file(video_path):
    """Locate the Tutorial 1 pose file for a video. Returns (path_or_None, note)."""
    target = f"{video_path.stem}{POSE_SUFFIX}"
    direct = POSE_DIR / target
    if direct.exists():
        return direct, ""
    nested = sorted(POSE_DIR.rglob(target))          # per-camera subfolders
    if len(nested) == 1:
        return nested[0], f"(in {nested[0].parent.name}/)"
    if len(nested) > 1:
        return nested[0], f"(WARNING: {len(nested)} files named {target}, used the first)"
    return None, f"no {target} found under POSE_DIR"


# ── Check the folders exist before blaming the contents ──────
for label, d in (("VIDEO_DIR", VIDEO_DIR), ("POSE_DIR", POSE_DIR)):
    if not d.exists():
        raise FileNotFoundError(
            f"{label} does not exist:\n    {d}\n\n"
            f"Edit section 1 at the top of this cell. If you have not run Tutorial 1 yet, "
            f"do that first -- it produces the *{POSE_SUFFIX} files this notebook needs."
        )

# Only now that the inputs check out do we create the output folder.
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Discover cameras ────────────────────────────────────────
VIDEO_PATHS, POSE_PATHS, notes = {}, {}, {}
skipped = []

all_videos = sorted(VIDEO_DIR.glob(VIDEO_GLOB))
for vp in all_videos:
    pp, note = find_pose_file(vp)
    if pp is None:
        skipped.append((vp.name, note))
        continue
    cam = camera_name(vp)
    if cam in VIDEO_PATHS:
        skipped.append((vp.name, f"camera label '{cam}' already used by "
                                 f"{VIDEO_PATHS[cam].name} -- tighten CAMERA_REGEX"))
        continue
    VIDEO_PATHS[cam], POSE_PATHS[cam], notes[cam] = vp, pp, note

CAMERAS = list(VIDEO_PATHS)

# ── Report ──────────────────────────────────────────────────
print("=" * 78)
print("CONFIGURATION")
print("=" * 78)
print(f"  VIDEO_DIR     {VIDEO_DIR}")
print(f"  POSE_DIR      {POSE_DIR}")
print(f"  OUT_DIR       {OUT_DIR.resolve()}")
print(f"  REID_WEIGHTS  {REID_WEIGHTS}  [{'found' if REID_WEIGHTS.exists() else 'NOT FOUND'}]")
print(f"  N_PEOPLE      {N_PEOPLE}")
print(f"  FRAME_STRIDE  {FRAME_STRIDE}"
      f"{'' if FRAME_STRIDE == 1 else f'  (using every {FRAME_STRIDE}th frame)'}")

print(f"\n{'=' * 78}")
print(f"CAMERAS FOUND: {len(CAMERAS)}   "
      f"(scanned {len(all_videos)} file(s) matching {VIDEO_GLOB!r})")
print("=" * 78)
if CAMERAS:
    w = max(len(c) for c in CAMERAS) + 2
    for cam in CAMERAS:
        print(f"  {cam:<{w}}{VIDEO_PATHS[cam].name}")
        print(f"  {'':<{w}}  -> {POSE_PATHS[cam].name} {notes[cam]}")

if skipped:
    print(f"\nSkipped {len(skipped)} file(s) -- normally harmless "
          f"(proxies, re-encodes, unrelated video):")
    for name, why in skipped[:10]:
        print(f"  - {name}\n      {why}")
    if len(skipped) > 10:
        print(f"  ... and {len(skipped) - 10} more")

# ── Fail clearly, or confirm we are good to go ──────────────
if not CAMERAS:
    raise FileNotFoundError(
        "No camera had both a video and a matching pose file.\n\n"
        f"  VIDEO_DIR   = {VIDEO_DIR}\n"
        f"  VIDEO_GLOB  = {VIDEO_GLOB!r}   -> matched {len(all_videos)} file(s)\n"
        f"  POSE_DIR    = {POSE_DIR}\n"
        f"  POSE_SUFFIX = {POSE_SUFFIX!r}\n\n"
        "Each video needs a pose file named <video_stem>" + POSE_SUFFIX + " under POSE_DIR.\n"
        "Check: (a) is VIDEO_GLOB the right extension? (b) did Tutorial 1 actually write\n"
        "the pose files, and under these exact names? (c) is POSE_SUFFIX right?"
    )

# Validate the skeleton against a real pose file.
_probe = sio.load_slp(str(POSE_PATHS[CAMERAS[0]]))
_nodes = _probe.skeletons[0].nodes
print(f"\nSkeleton in {POSE_PATHS[CAMERAS[0]].name}: {len(_nodes)} nodes")
if max(TORSO_IDX) >= len(_nodes):
    raise ValueError(
        f"TORSO_IDX {TORSO_IDX} is out of range for a {len(_nodes)}-node skeleton.\n"
        f"Nodes: {[n.name for n in _nodes]}\n"
        "Update TORSO_IDX in section 4 to your shoulder/hip indices."
    )
print(f"  torso nodes -> {[_nodes[i].name for i in TORSO_IDX]}")
if len(_nodes) != len(NODE_NAMES):
    print(f"  NOTE: NODE_NAMES has {len(NODE_NAMES)} entries but this skeleton has "
          f"{len(_nodes)}. Only TORSO_IDX actually matters, so this is usually fine.")

print()
print("=" * 78)
problems = []
if not REID_WEIGHTS.exists():
    problems.append(f"ReID weights not found at {REID_WEIGHTS} -- STEP 5 will fail.")
if len(CAMERAS) < 2:
    problems.append("Only one camera: Stages 1-2 will work, but Stage 3 "
                    "(cross-camera identities) needs at least two.")
if problems:
    print("BEFORE YOU CONTINUE:")
    for p in problems:
        print(f"  ! {p}")
else:
    print(f"READY -- {len(CAMERAS)} cameras, {N_PEOPLE} people to resolve. Run the next cells.")
print("=" * 78)


---

## Part 4: Torso / Shirt Cropping

For robust ReID in a classroom or camp setting, we crop just the **torso region**
(shoulders to hips) rather than the whole body.

### Keypoints Used

```
   left_shoulder(5) ──── right_shoulder(6)
         |                      |
         |    TORSO REGION      |
         |                      |
     left_hip(11) ──────── right_hip(12)
```

### Why torso and not full body?

- **Legs are usually occluded** by desks, chairs and other kids — a full-body box is mostly
  furniture, and furniture looks the same for everybody.
- **Shirt colour and pattern are the most discriminative signal** available in this scene.
- A tighter box means less background bleeding into the embedding.

### Two details that matter more than they look

**1. Preserve aspect ratio.** OSNet was trained on **128×256** (W×H) person crops — a 1:2
portrait aspect. A torso box is often *wider* than it is tall. If you `cv2.resize` that
straight to 128×256 you horizontally squash every crop, and the amount of squashing depends
on the person's pose. That is appearance variation you are injecting yourself, and the
encoder cannot tell it apart from a genuine change of clothing.

So instead we **expand the box to a 1:2 aspect around its centre** before cropping — which
also pulls in a little head and hip context for free — and letterbox-pad if the expanded box
runs off the edge of the frame.

**2. Crop in memory.** An earlier version of this notebook wrote every crop to disk as a
JPEG — one file per detection per camera, which runs into six figures fast. We stream
crops through the encoder in batches instead:

- no filesystem round-trip, no JPEG re-compression loss
- the `(camera, frame_idx, instance_idx)` join key stays attached in memory
- set `SAVE_CROPS = True` below if you want them on disk for visual debugging anyway

In [ ]:
# ============================================================
# STEP 4: Torso cropping — the functions
# ============================================================
# We define the cropping here and preview it on a few frames. The full pass over
# all cameras happens in STEP 7, where crops are fed straight into the encoder and
# discarded batch by batch.
#
# Why not crop everything up front? 126,000 crops x 128x256x3 bytes is ~12 GB.
# Embeddings for the same crops are 126,000 x 512 x 4 bytes = ~260 MB. So we keep
# the embeddings and throw the pixels away as we go.


def torso_box(pts, pad=TORSO_PAD, target_aspect=CROP_W / CROP_H):
    """Torso bounding box from COCO keypoints, expanded to `target_aspect` (W/H).

    Falls back to all visible keypoints when too few torso joints are visible.
    Returns (x0, y0, x1, y1) as ints, or None if unusable.
    """
    torso = pts[TORSO_IDX]
    vis = ~np.isnan(torso[:, 0])
    if vis.sum() >= 2:
        src = torso[vis]
    else:
        # Torso occluded — fall back to whatever keypoints we do have, so the
        # instance still gets an embedding instead of being silently dropped.
        allvis = ~np.isnan(pts[:, 0])
        if allvis.sum() < 2:
            return None
        src = pts[allvis]

    x0, y0 = float(src[:, 0].min()), float(src[:, 1].min())
    x1, y1 = float(src[:, 0].max()), float(src[:, 1].max())

    # Pad proportionally to the box size
    w, h = max(x1 - x0, 1.0), max(y1 - y0, 1.0)
    x0, x1 = x0 - pad * w, x1 + pad * w
    y0, y1 = y0 - pad * h, y1 + pad * h
    w, h = x1 - x0, y1 - y0

    # Expand (never shrink) to the target aspect around the centre, so the resize
    # in crop_letterbox does not distort the person.
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    if w / h > target_aspect:
        h = w / target_aspect          # too wide -> grow height
    else:
        w = h * target_aspect          # too tall -> grow width
    x0, x1 = cx - w / 2, cx + w / 2
    y0, y1 = cy - h / 2, cy + h / 2

    if (x1 - x0) < 8 or (y1 - y0) < 8:
        return None
    return int(round(x0)), int(round(y0)), int(round(x1)), int(round(y1))


def crop_letterbox(frame, box, out_w=CROP_W, out_h=CROP_H):
    """Crop `box` from `frame`, zero-padding whatever falls outside, then resize.

    Padding rather than clamping keeps the person centred and the aspect intact even
    when the box hangs off the edge of the image.
    """
    x0, y0, x1, y1 = box
    fh, fw = frame.shape[:2]
    canvas = np.zeros((y1 - y0, x1 - x0, 3), dtype=frame.dtype)

    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(fw, x1), min(fh, y1)
    if sx1 <= sx0 or sy1 <= sy0:
        return None                     # box entirely outside the frame
    canvas[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = frame[sy0:sy1, sx0:sx1]
    return cv2.resize(canvas, (out_w, out_h), interpolation=cv2.INTER_LINEAR)


def iter_detections(camera, stride=FRAME_STRIDE):
    """Yield (frame_idx, instance_idx, points, crop) for one camera, streaming.

    Decodes the video sequentially (much faster than seeking) and only touches frames
    that actually have detections.
    """
    labels = sio.load_slp(str(POSE_PATHS[camera]))
    by_frame = {}
    for lf in labels:
        if lf.frame_idx % stride:
            continue
        by_frame[lf.frame_idx] = [(i, inst.numpy()) for i, inst in enumerate(lf.instances)]

    cap = cv2.VideoCapture(str(VIDEO_PATHS[camera]))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {VIDEO_PATHS[camera]}")
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    try:
        for frame_idx in range(n_frames):
            ret, frame = cap.read()
            if not ret:
                break
            dets = by_frame.get(frame_idx)
            if not dets:
                continue
            for inst_idx, pts in dets:
                box = torso_box(pts)
                if box is None:
                    continue
                crop = crop_letterbox(frame, box)
                if crop is None:
                    continue
                yield frame_idx, inst_idx, pts, crop
    finally:
        cap.release()


# ── Preview: what do these crops actually look like? ─────────
import matplotlib.pyplot as plt

PREVIEW_CAM = CAMERAS[0]
preview = []
for frame_idx, inst_idx, pts, crop in iter_detections(PREVIEW_CAM, stride=1):
    preview.append((frame_idx, inst_idx, crop))
    if len(preview) >= 12:
        break

if preview:
    fig, axes = plt.subplots(1, len(preview), figsize=(1.6 * len(preview), 4))
    axes = np.atleast_1d(axes)
    for ax, (f, i, crop) in zip(axes, preview):
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"f{f} i{i}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"{PREVIEW_CAM} torso crops — {CROP_W}x{CROP_H}, aspect preserved", fontsize=12)
    plt.tight_layout()
    plt.show()
    print("Check these before going further: each crop should be a recognisable torso, "
          "upright, not horizontally squashed.")
else:
    print(f"No crops produced for {PREVIEW_CAM} — check the pose .slp and video paths in STEP 3.")

---

## Part 5: Extracting ReID Embeddings

### What is an Embedding?

An **embedding** is a fixed-length vector (512 dimensions for OSNet) that encodes the visual
appearance of a person. The property we rely on:

- **Same person** in different cameras -> embeddings are **close** (high cosine similarity)
- **Different people** -> embeddings are **far apart** (low cosine similarity)

That is the whole basis of ReID. How well it holds *for your scene* is an empirical question,
which is what Part 6 measures rather than assumes.

### Getting the encoder

We use the OSNet encoder that ships inside BoxMOT. The catch: BoxMOT has moved it between
attributes across releases (`.encoder`, `.model`, `.model.model`, and a separate
`ReidAutoBackend` wrapper), so hard-coding one path breaks on a version bump.

STEP 5 instead **probes the candidates and validates each with a dummy forward pass**, keeping
the first that returns a sensible `(batch, dim)` feature matrix. If none work it falls back to
building `osnet_x0_25` through `torchreid` directly. It prints which path it used, so you
always know what produced your embeddings.

### The pipeline per crop

1. Resize to 256x128 (H x W) — already done by the cropping in STEP 4
2. BGR -> RGB, scale to [0,1], normalize with ImageNet mean/std
3. Forward through OSNet -> 512-dim vector
4. **L2-normalize**, so cosine similarity is a plain dot product

STEP 6 does all four for a whole batch at once with tensor ops rather than per-image PIL
calls — worth it when you are embedding 10^5 crops.


In [ ]:
# ============================================================
# STEP 5: Initialize the ReID encoder (OSNet)
# ============================================================
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: no GPU detected. Embedding ~126k crops on CPU will take a long time.\n"
          "         Consider raising FRAME_STRIDE in STEP 3 for a development pass.")

assert REID_WEIGHTS.exists(), f"ReID weights not found: {REID_WEIGHTS}"


def load_reid_encoder(weights, device):
    """Return a callable `f(batch_tensor) -> (B, D) embeddings`.

    BoxMOT's internal layout has moved around between releases (`.encoder`, `.model`,
    `.model.model`, and a `ReidAutoBackend` wrapper), so rather than pinning one
    attribute we probe for something that behaves like a feature extractor and
    validate it with a dummy forward pass.
    """
    candidates = []

    try:
        from boxmot.trackers.strongsort.strongsort import StrongSort
        tracker = StrongSort(reid_weights=weights, device=device, half=False)
        for attr in ("encoder", "model"):
            obj = getattr(tracker, attr, None)
            if obj is None:
                continue
            candidates.append((f"StrongSort.{attr}", obj))
            inner = getattr(obj, "model", None)
            if inner is not None:
                candidates.append((f"StrongSort.{attr}.model", inner))
    except Exception as e:                                     # noqa: BLE001
        print(f"  StrongSort path unavailable: {type(e).__name__}: {e}")

    try:  # BoxMOT >= 11 exposes the backend directly
        from boxmot.appearance.reid_auto_backend import ReidAutoBackend
        backend = ReidAutoBackend(weights=weights, device=device, half=False)
        candidates.append(("ReidAutoBackend.model", backend.model))
    except Exception:                                          # noqa: BLE001
        pass

    try:  # last resort: torchreid directly
        import torchreid
        m = torchreid.models.build_model("osnet_x0_25", num_classes=1, pretrained=False)
        torchreid.utils.load_pretrained_weights(m, str(weights))
        m.eval().to(device)
        candidates.append(("torchreid.osnet_x0_25", m))
    except Exception:                                          # noqa: BLE001
        pass

    dummy = torch.zeros(2, 3, CROP_H, CROP_W, device=device)
    for name, obj in candidates:
        fn = obj if callable(obj) else getattr(obj, "forward", None)
        if fn is None:
            continue
        try:
            with torch.inference_mode():
                out = fn(dummy)
            if isinstance(out, (list, tuple)):
                out = out[0]
            out = torch.as_tensor(out)
            if out.ndim == 2 and out.shape[0] == 2 and out.shape[1] >= 64:
                print(f"  Using encoder: {name}  ->  embedding dim {out.shape[1]}")
                return fn, int(out.shape[1])
        except Exception as e:                                 # noqa: BLE001
            print(f"  {name} rejected: {type(e).__name__}")

    raise RuntimeError(
        "Could not obtain a working ReID encoder. Tried: "
        + ", ".join(n for n, _ in candidates)
        + "\nCheck `pip install boxmot` (or torchreid) and that the weights file matches "
          "the osnet_x0_25 architecture."
    )


encoder, EMBED_DIM = load_reid_encoder(REID_WEIGHTS, device)
print(f"\nReID encoder ready — {EMBED_DIM}-dim embeddings from {REID_WEIGHTS.name}")

In [ ]:
# ============================================================
# STEP 6: Preprocessing — numpy crops -> normalized tensor batch
# ============================================================
# OSNet expects:
#   - 256 x 128 (H x W) RGB
#   - float in [0,1], then ImageNet mean/std normalization
#   - NCHW tensor layout
#
# Our crops are already CROP_H x CROP_W BGR uint8 from cv2, so we do the whole
# batch with numpy/torch ops rather than PIL per image — roughly 20x faster and it
# keeps the crop -> embedding path free of per-image Python overhead.

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def preprocess_batch(crops_bgr):
    """(B, H, W, 3) uint8 BGR  ->  (B, 3, H, W) normalized float tensor on `device`."""
    x = torch.from_numpy(np.ascontiguousarray(crops_bgr)).to(device)
    x = x[..., [2, 1, 0]]                      # BGR -> RGB
    x = x.permute(0, 3, 1, 2).float().div_(255.0)
    return (x - IMAGENET_MEAN) / IMAGENET_STD


@torch.inference_mode()
def embed_batch(crops_bgr):
    """(B, H, W, 3) uint8 BGR  ->  (B, D) L2-normalized float32 embeddings."""
    out = encoder(preprocess_batch(crops_bgr))
    if isinstance(out, (list, tuple)):
        out = out[0]
    feats = torch.as_tensor(out).float()
    feats = torch.nn.functional.normalize(feats, p=2, dim=1)
    return feats.cpu().numpy()


# ── Sanity check on the preview crops from STEP 4 ────────────
if preview:
    test_batch = np.stack([c for _, _, c in preview])
    t = preprocess_batch(test_batch)
    f = embed_batch(test_batch)
    print(f"Input batch:   {tuple(test_batch.shape)}  uint8 BGR")
    print(f"Tensor:        {tuple(t.shape)}  {t.dtype}  "
          f"range [{t.min():.2f}, {t.max():.2f}]")
    print(f"Embeddings:    {f.shape}  {f.dtype}")
    print(f"L2 norms:      {np.linalg.norm(f, axis=1).round(4)}   (should all be 1.0)")

    # Adjacent crops in the preview are consecutive detections, so a few of these
    # pairs are the same person one frame apart -- their similarity should be high.
    sim = f @ f.T
    print(f"\nSelf-similarity matrix (first 6x6):\n{sim[:6, :6].round(3)}")
else:
    print("No preview crops available — re-run STEP 4.")

In [ ]:
# ============================================================
# STEP 7: Embed every detection in every camera (streaming)
# ============================================================
# One sequential pass per camera: crop -> buffer -> encode -> keep only embeddings.
# Nothing is sampled; every detection that produced a usable crop gets an embedding,
# because we need a decision for every instance we are going to write back to .slp.

EMB_CACHE = OUT_DIR / "embeddings.npz"
RECOMPUTE = False            # True forces a re-run even if the cache exists


def embed_camera(camera):
    """Return (feats (N,D), keys (N,2), centroids (N,2)) for one camera."""
    feats_out, keys_out, cents_out = [], [], []
    buf_crops, buf_keys, buf_cents = [], [], []

    def flush():
        if not buf_crops:
            return
        feats_out.append(embed_batch(np.stack(buf_crops)))
        keys_out.extend(buf_keys)
        cents_out.extend(buf_cents)
        buf_crops.clear(); buf_keys.clear(); buf_cents.clear()

    for frame_idx, inst_idx, pts, crop in tqdm(
        iter_detections(camera), desc=f"  {camera}", leave=False, unit="det"
    ):
        buf_crops.append(crop)
        buf_keys.append((frame_idx, inst_idx))
        # Centroid of visible keypoints — the motion cue for tracklet linking.
        buf_cents.append(np.nanmean(pts, axis=0))
        if len(buf_crops) >= BATCH_SIZE:
            flush()
    flush()

    feats = np.concatenate(feats_out) if feats_out else np.zeros((0, EMBED_DIM), np.float32)
    keys = np.array(keys_out, dtype=int).reshape(-1, 2)
    cents = np.array(cents_out, dtype=float).reshape(-1, 2)
    return feats, keys, cents


cam_feats, cam_keys, cam_cents = {}, {}, {}

if EMB_CACHE.exists() and not RECOMPUTE:
    print(f"Loading cached embeddings from {EMB_CACHE}")
    z = np.load(EMB_CACHE)
    for camera in CAMERAS:
        cam_feats[camera] = z[f"{camera}_feats"]
        cam_keys[camera] = z[f"{camera}_keys"]
        cam_cents[camera] = z[f"{camera}_cents"]
else:
    print(f"Embedding all detections (stride={FRAME_STRIDE}, batch={BATCH_SIZE})\n")
    for camera in CAMERAS:
        cam_feats[camera], cam_keys[camera], cam_cents[camera] = embed_camera(camera)
    np.savez_compressed(
        EMB_CACHE,
        **{f"{c}_feats": cam_feats[c] for c in CAMERAS},
        **{f"{c}_keys": cam_keys[c] for c in CAMERAS},
        **{f"{c}_cents": cam_cents[c] for c in CAMERAS},
    )
    print(f"\nCached to {EMB_CACHE}")

print(f"\n{'camera':<8}{'detections':>12}{'frames':>9}{'dim':>6}   mean dets/frame")
for camera in CAMERAS:
    k = cam_keys[camera]
    n_frames = len(np.unique(k[:, 0])) if len(k) else 0
    per_frame = len(k) / n_frames if n_frames else 0
    print(f"{camera:<8}{len(k):>12,}{n_frames:>9}{cam_feats[camera].shape[1]:>6}"
          f"   {per_frame:>5.2f}")

total = sum(len(cam_keys[c]) for c in CAMERAS)
print(f"\nTotal embedded detections: {total:,}")
print(f"Embedding memory: {sum(f.nbytes for f in cam_feats.values())/1e6:.0f} MB")

---

## Part 6: Cosine Similarity — and What It Actually Buys Us

### What is cosine similarity?

Cosine similarity measures the angle between two vectors:

```
cosine_sim(A, B) = (A . B) / (||A|| x ||B||)
```

Our embeddings are L2-normalized (`||A|| = ||B|| = 1`), so this is just a dot product —
which means the whole all-pairs similarity matrix is one matrix multiply: `F @ F.T`.

### Read the distribution before you trust a threshold

You will see advice like "> 0.6 means same person". Treat any such number as a
**hypothesis about your data, not a fact about the metric**. The right threshold depends on
the encoder, the crop style, the lighting, and how similar the people in your scene happen to
look. Six kids at summer camp, several in the same camp T-shirt, is close to the worst case.

So the cell below splits the pairs into three kinds instead of pooling them:

| Pair type | What it tells you |
|---|---|
| **Same camera, nearby frames** | The easy case. Should be very high — if it isn't, cropping or the encoder is broken. |
| **Same camera, distant frames** | Robustness to pose and lighting change over time. |
| **Different cameras, same frame** | **The one that matters.** This is the actual ReID task: same person, different viewpoint. |

An aggregate histogram over random pairs — what this notebook used to plot — hides exactly
the comparison you care about, because the overwhelming majority of random pairs are
different-person pairs and they swamp the distribution.

### The separation you're looking for

If "different cameras, same frame" similarities overlap heavily with random different-person
similarities, then appearance alone cannot solve this scene, and you need the geometry
(epipolar constraints from `calibration.toml`) to disambiguate. That is a legitimate outcome
worth discovering *before* you trust the identity assignment.

In [ ]:
# ============================================================
# STEP 8: Similarity structure, broken down by pair type
# ============================================================
rng = np.random.default_rng(42)
N_PAIRS = 20000


def index_by_frame(camera):
    """frame_idx -> array of row indices into cam_feats[camera]."""
    out = {}
    for row, (f, _i) in enumerate(cam_keys[camera]):
        out.setdefault(int(f), []).append(row)
    return {f: np.array(v) for f, v in out.items()}


frame_index = {c: index_by_frame(c) for c in CAMERAS}

# ── 1. Same camera, nearby frames (<= 2 apart) ───────────────
same_cam_near = []
for _ in range(N_PAIRS):
    c = CAMERAS[rng.integers(len(CAMERAS))]
    k = cam_keys[c]
    if len(k) < 2:
        continue
    a = rng.integers(len(k))
    fa = k[a, 0]
    # candidate partners within +/-2 frames, excluding the same detection
    cand = [r for df in (-2, -1, 1, 2) for r in frame_index[c].get(int(fa + df), [])]
    if not cand:
        continue
    b = cand[rng.integers(len(cand))]
    same_cam_near.append(float(cam_feats[c][a] @ cam_feats[c][b]))

# ── 2. Same camera, distant frames (>= 500 apart) ────────────
same_cam_far = []
for _ in range(N_PAIRS):
    c = CAMERAS[rng.integers(len(CAMERAS))]
    k = cam_keys[c]
    if len(k) < 2:
        continue
    a, b = rng.integers(len(k)), rng.integers(len(k))
    if abs(int(k[a, 0]) - int(k[b, 0])) < 500:
        continue
    same_cam_far.append(float(cam_feats[c][a] @ cam_feats[c][b]))

# ── 3. Different cameras, same frame ─────────────────────────
# NOTE: these are a MIX of same-person and different-person pairs -- we have no
# identities yet, that is the whole point of the next section. What matters is the
# upper tail: if the true cross-view matches were similar, some pairs must score high.
cross_cam = []
for _ in range(N_PAIRS):
    ca, cb = rng.choice(len(CAMERAS), size=2, replace=False)
    ca, cb = CAMERAS[ca], CAMERAS[cb]
    shared = set(frame_index[ca]) & set(frame_index[cb])
    if not shared:
        continue
    f = list(shared)[rng.integers(len(shared))]
    ra = frame_index[ca][f][rng.integers(len(frame_index[ca][f]))]
    rb = frame_index[cb][f][rng.integers(len(frame_index[cb][f]))]
    cross_cam.append(float(cam_feats[ca][ra] @ cam_feats[cb][rb]))

groups = {
    "same cam, <=2 frames apart": np.array(same_cam_near),
    "same cam, >=500 frames apart": np.array(same_cam_far),
    "different cams, same frame": np.array(cross_cam),
}

print(f"{'pair type':<32}{'n':>7}{'mean':>8}{'p05':>8}{'p50':>8}{'p95':>8}")
for name, v in groups.items():
    if len(v) == 0:
        print(f"{name:<32}{'0':>7}   (no pairs sampled)")
        continue
    print(f"{name:<32}{len(v):>7}{v.mean():>8.3f}"
          f"{np.percentile(v, 5):>8.3f}{np.percentile(v, 50):>8.3f}"
          f"{np.percentile(v, 95):>8.3f}")

print("\nHow to read this:")
print("  * 'same cam, <=2 frames' should be clearly the highest — it is the same person,")
print("    same view, ~30 ms apart. If it is not high, fix cropping before continuing.")
print("  * 'different cams, same frame' mixes same- and different-person pairs. Its upper")
print("    tail is where the true cross-view matches live.")
print("  * If the upper tail of group 3 does not exceed the bulk of group 2, appearance")
print("    alone will struggle and you should add epipolar gating.")

In [ ]:
# ============================================================
# STEP 9: Visualize the similarity distributions
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ── Left: overlaid histograms per pair type ──────────────────
ax = axes[0]
colors = {"same cam, <=2 frames apart": "#2a9d8f",
          "same cam, >=500 frames apart": "#e9c46a",
          "different cams, same frame": "#e76f51"}
for name, v in groups.items():
    if len(v) == 0:
        continue
    ax.hist(v, bins=80, range=(-0.2, 1.0), alpha=0.55, density=True,
            label=f"{name}  (n={len(v):,})", color=colors[name], edgecolor="none")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Density")
ax.set_title("Similarity by pair type — the separation is what matters")
ax.legend(fontsize=9, loc="upper left")
ax.grid(alpha=0.2)

# ── Right: one frame's cross-camera similarity matrix ────────
# Concrete and much more legible than a 50x50 block of random crops: for a single
# frame, every detection in every camera against every other.
ax = axes[1]
probe_frame = None
for f in sorted(frame_index[CAMERAS[0]]):
    if sum(len(frame_index[c].get(f, [])) for c in CAMERAS) >= 2 * len(CAMERAS):
        probe_frame = f
        break

if probe_frame is not None:
    rows, labels_ = [], []
    for c in CAMERAS:
        for r in frame_index[c].get(probe_frame, []):
            rows.append(cam_feats[c][r])
            labels_.append(f"{c[-1]}:{cam_keys[c][r, 1]}")
    F = np.stack(rows)
    S = F @ F.T
    im = ax.imshow(S, cmap="RdYlBu_r", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels_)), labels_, rotation=90, fontsize=7)
    ax.set_yticks(range(len(labels_)), labels_, fontsize=7)
    ax.set_title(f"Frame {probe_frame}: all detections, all cameras\n"
                 f"(label = camera:instance)", fontsize=11)
    plt.colorbar(im, ax=ax, label="Cosine similarity", fraction=0.046)
    print(f"Probe frame {probe_frame}: {len(labels_)} detections across {len(CAMERAS)} cameras.")
    print("Look for bright off-diagonal blocks — those are the same person seen from")
    print("different cameras, and they are what the identity assignment has to find.")
else:
    ax.text(0.5, 0.5, "No frame with detections in all cameras", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "reid_similarity_analysis.png", dpi=150)
plt.show()
print(f"\nSaved: {OUT_DIR / 'reid_similarity_analysis.png'}")

---

## Part 7: From Embeddings to Identities

Everything so far produced *descriptions* of detections. Now we have to make **decisions**:
every instance gets exactly one identity label, because that is what a SLEAP `Track` is.

### Why not just threshold the similarity matrix?

The obvious approach — "for each detection in CAM1, find its most similar detection in CAM2
and call them the same person" — fails in three ways:

1. **It isn't a valid assignment.** Two different people in CAM1 can both pick the same
   best match in CAM2. Physically impossible, but nothing stops it.
2. **It ignores time.** Appearance alone is noisiest exactly when it matters (motion blur,
   turning away, partial occlusion). Frame-to-frame *position* is a strong, nearly free cue
   that appearance-only matching throws away.
3. **It ignores the mutual-exclusion constraint.** Two detections in the *same frame* of the
   *same camera* are definitionally different people. That is hard information.

### The three stages

```
   per-detection embeddings
             │
             ▼
  ┌──────────────────────────┐
  │ Stage 1: tracklets       │   Hungarian per frame, cost = appearance + motion
  │ (within camera, in time) │   -> many short, high-confidence fragments
  └──────────────────────────┘
             │
             ▼
  ┌──────────────────────────┐
  │ Stage 2: consolidate     │   merge fragments into exactly N_PEOPLE identities,
  │ (within camera)          │   never merging two that overlap in time
  └──────────────────────────┘
             │
             ▼
  ┌──────────────────────────┐
  │ Stage 3: global IDs      │   Hungarian between each camera's gallery and a
  │ (across cameras)         │   running global gallery -> person_0 .. person_N
  └──────────────────────────┘
```

**Stage 1 — tracklets.** For each frame, build a cost matrix between currently-active
tracklets and the new detections:

```
cost = W_APPEARANCE * (1 - cosine_sim)  +  W_MOTION * (centroid_move / max_move)
```

then solve it with `scipy.optimize.linear_sum_assignment` (the Hungarian algorithm), which
returns a genuine one-to-one assignment. Links costing more than `MAX_LINK_COST` are rejected
and start a new tracklet instead — it is much better to produce two clean fragments than one
fragment with a swap buried in it.

**Stage 2 — consolidation.** Tracklets fragment whenever someone is occluded or leaves the
frame. We merge them by average-embedding similarity, greedily, subject to one hard
constraint: **two tracklets that are ever present in the same frame cannot be the same
person.** That constraint is what makes this reliable — it is pure geometry, no appearance
guessing, and it prevents the single most common failure mode.

**Stage 3 — global IDs.** Each camera now has `N_PEOPLE` local identities with a gallery
embedding each. We walk the cameras, Hungarian-matching each camera's gallery against a
running global gallery, and update the global gallery as a weighted mean.

### The assumption in Stage 3, stated plainly

Hungarian matching on galleries assumes **all `N_PEOPLE` are visible in every camera**. When
someone is missing from a view, a 1-to-1 assignment is forced to pair them with *somebody*,
and that pairing will be wrong. For this clip (a fixed group in a shared space) the assumption
mostly holds, which is why it works well enough to be worth proofreading. The principled fix
is epipolar gating from `calibration.toml` — forbid matches whose 3D rays don't intersect —
and that is the natural next extension.

This is exactly why the output goes to SLEAP for **human review** rather than straight into
triangulation.

In [ ]:
# ============================================================
# STEP 10: Stage 1 — build within-camera tracklets
# ============================================================
from scipy.optimize import linear_sum_assignment

BIG = 1e6      # cost used to forbid an assignment outright


def build_tracklets(camera, verbose=True):
    """Link detections into tracklets using appearance + motion.

    Returns
    -------
    tracklet_id : (N,) int array, parallel to cam_feats[camera]; -1 = unassigned
    tracklets   : dict id -> {"rows": [...], "frames": [...]}
    """
    feats = cam_feats[camera]
    keys = cam_keys[camera]
    cents = cam_cents[camera]
    n = len(keys)
    tracklet_id = np.full(n, -1, dtype=int)
    if n == 0:
        return tracklet_id, {}

    # Frame diagonal sets the motion scale, so MAX_MOVE_FRAC is resolution-independent.
    cap = cv2.VideoCapture(str(VIDEO_PATHS[camera]))
    fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1920
    fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080
    cap.release()
    max_move = MAX_MOVE_FRAC * float(np.hypot(fw, fh))

    rows_by_frame = {}
    for row, f in enumerate(keys[:, 0]):
        rows_by_frame.setdefault(int(f), []).append(row)

    # active: id -> {"emb": EMA embedding, "cent": last centroid, "last": last frame}
    active = {}
    tracklets = {}
    next_id = 0
    n_links = n_new = 0

    for frame in sorted(rows_by_frame):
        rows = rows_by_frame[frame]

        # Retire tracklets we have not seen in a while.
        for tid in [t for t, s in active.items() if frame - s["last"] > MAX_AGE]:
            del active[tid]

        if active:
            aid = list(active)
            A = np.stack([active[t]["emb"] for t in aid])            # (n_active, D)
            AC = np.stack([active[t]["cent"] for t in aid])          # (n_active, 2)
            D = feats[rows]                                          # (n_det, D)
            DC = cents[rows]

            app = 1.0 - (A @ D.T)                                    # (n_active, n_det)
            move = np.linalg.norm(AC[:, None, :] - DC[None, :, :], axis=-1)
            gap = np.array([[max(1, frame - active[t]["last"])] for t in aid])
            motion = move / (max_move * gap)                         # allow more drift after a gap

            cost = W_APPEARANCE * app + W_MOTION * np.minimum(motion, 1.0)
            cost[motion > 1.0] = BIG                                 # impossible jump
            cost[cost > MAX_LINK_COST] = BIG                         # too weak to trust

            ri, ci = linear_sum_assignment(cost)
            taken = set()
            for r, c in zip(ri, ci):
                if cost[r, c] >= BIG:
                    continue
                tid = aid[r]
                row = rows[c]
                tracklet_id[row] = tid
                tracklets[tid]["rows"].append(row)
                tracklets[tid]["frames"].append(frame)
                # EMA keeps the gallery stable without letting it go stale.
                active[tid]["emb"] = 0.9 * active[tid]["emb"] + 0.1 * feats[row]
                active[tid]["emb"] /= np.linalg.norm(active[tid]["emb"]) + 1e-12
                active[tid]["cent"] = cents[row]
                active[tid]["last"] = frame
                taken.add(c)
                n_links += 1
            unmatched = [rows[c] for c in range(len(rows)) if c not in taken]
        else:
            unmatched = rows

        for row in unmatched:
            tid = next_id
            next_id += 1
            tracklet_id[row] = tid
            tracklets[tid] = {"rows": [row], "frames": [frame]}
            active[tid] = {"emb": feats[row].copy(), "cent": cents[row], "last": frame}
            n_new += 1

    if verbose:
        lens = np.array([len(t["rows"]) for t in tracklets.values()])
        print(f"{camera}: {len(tracklets):>5} tracklets  "
              f"({n_links:,} links, {n_new:,} starts)   "
              f"len median {int(np.median(lens))}, max {lens.max()}, "
              f"{(lens < MIN_TRACKLET_LEN).sum()} shorter than {MIN_TRACKLET_LEN}")
    return tracklet_id, tracklets


cam_tracklet_id, cam_tracklets = {}, {}
print("Stage 1 — within-camera tracklets "
      f"(W_app={W_APPEARANCE}, W_mot={W_MOTION}, max_cost={MAX_LINK_COST})\n")
for camera in CAMERAS:
    cam_tracklet_id[camera], cam_tracklets[camera] = build_tracklets(camera)

print("\nMany short tracklets is EXPECTED and healthy — Stage 2 merges them. A small number")
print("of long tracklets would mean the linker is being reckless and burying ID swaps.")

In [ ]:
# ============================================================
# STEP 11: Stage 2 — consolidate tracklets into N_PEOPLE identities
# ============================================================
# Greedy agglomerative merging by gallery similarity, with one hard constraint:
# two tracklets that ever appear in the SAME FRAME are different people and can
# never be merged. That single rule does most of the work.


def consolidate(camera, n_people=N_PEOPLE, verbose=True):
    """Merge tracklets into <= n_people clusters. Returns (N,) local person id, -1 = dropped."""
    feats = cam_feats[camera]
    keys = cam_keys[camera]
    tracklets = cam_tracklets[camera]
    local_id = np.full(len(keys), -1, dtype=int)
    if not tracklets:
        return local_id

    # Keep only tracklets long enough to have a trustworthy mean embedding.
    tids = [t for t, v in tracklets.items() if len(v["rows"]) >= MIN_TRACKLET_LEN]
    n_dropped_tracklets = len(tracklets) - len(tids)
    if not tids:
        tids = list(tracklets)
        n_dropped_tracklets = 0

    n_frames_total = int(keys[:, 0].max()) + 1

    # Cluster state: mean embedding, frame-occupancy mask, member rows.
    clusters = []
    for t in tids:
        rows = np.array(tracklets[t]["rows"])
        emb = feats[rows].mean(axis=0)
        emb /= np.linalg.norm(emb) + 1e-12
        occ = np.zeros(n_frames_total, dtype=bool)
        occ[np.array(tracklets[t]["frames"])] = True
        clusters.append({"emb": emb, "occ": occ, "rows": list(rows), "n": len(rows)})

    # Greedily merge the most similar compatible pair until we hit n_people.
    n_merges = 0
    while len(clusters) > n_people:
        E = np.stack([c["emb"] for c in clusters])
        S = E @ E.T
        np.fill_diagonal(S, -np.inf)

        # Forbid pairs that co-occur in any frame.
        order = np.argsort(S.ravel())[::-1]
        merged = False
        for flat in order:
            i, j = divmod(int(flat), len(clusters))
            if i >= j or not np.isfinite(S[i, j]):
                continue
            if (clusters[i]["occ"] & clusters[j]["occ"]).any():
                continue                      # same frame -> different people
            a, b = clusters[i], clusters[j]
            na, nb = a["n"], b["n"]
            emb = (a["emb"] * na + b["emb"] * nb) / (na + nb)
            emb /= np.linalg.norm(emb) + 1e-12
            clusters[i] = {"emb": emb, "occ": a["occ"] | b["occ"],
                           "rows": a["rows"] + b["rows"], "n": na + nb}
            clusters.pop(j)
            merged = True
            n_merges += 1
            break
        if not merged:
            # Every remaining pair overlaps in time: more simultaneous people than
            # n_people. Stop rather than force a physically impossible merge.
            if verbose:
                print(f"  {camera}: stopped at {len(clusters)} clusters — all remaining "
                      f"pairs overlap in time (more than {n_people} people visible at once?)")
            break

    # Largest cluster becomes person_0, so ordering is stable across re-runs.
    clusters.sort(key=lambda c: -c["n"])
    for pid, c in enumerate(clusters[:n_people]):
        local_id[np.array(c["rows"], dtype=int)] = pid

    if verbose:
        assigned = (local_id >= 0).sum()
        sizes = [c["n"] for c in clusters[:n_people]]
        print(f"{camera}: {len(clusters)} identities  ({n_merges} merges, "
              f"{n_dropped_tracklets} short tracklets dropped)   "
              f"{assigned:,}/{len(keys):,} detections assigned ({assigned/max(len(keys),1):.1%})")
        print(f"         cluster sizes: {sizes}")
    return local_id


cam_local_id = {}
print(f"Stage 2 — consolidating to {N_PEOPLE} identities per camera\n")
for camera in CAMERAS:
    cam_local_id[camera] = consolidate(camera)

print("\nDetections left at -1 belong to dropped short tracklets or to a 7th+ person. They")
print("are still written to the .slp, just without a track — visible for you to fix by hand.")

In [ ]:
# ============================================================
# STEP 12: Stage 3 — link identities across cameras into global IDs
# ============================================================
# Each camera has N_PEOPLE local identities. We need to know that CAM1's local #2 and
# CAM3's local #0 are the same human, so that both get the track name `person_2`.


def gallery(camera):
    """(n_people, D) mean embedding per local identity, plus support counts."""
    feats, lid = cam_feats[camera], cam_local_id[camera]
    G = np.zeros((N_PEOPLE, feats.shape[1]), dtype=np.float64)
    counts = np.zeros(N_PEOPLE, dtype=int)
    for pid in range(N_PEOPLE):
        rows = np.where(lid == pid)[0]
        counts[pid] = len(rows)
        if len(rows):
            g = feats[rows].mean(axis=0)
            G[pid] = g / (np.linalg.norm(g) + 1e-12)
    return G, counts


galleries = {c: gallery(c) for c in CAMERAS}

# Anchor on the camera with the most confidently-assigned detections.
anchor = max(CAMERAS, key=lambda c: galleries[c][1].sum())
print(f"Anchor camera: {anchor} (most assigned detections)\n")

global_gallery = galleries[anchor][0].copy()
global_support = galleries[anchor][1].astype(float).copy()
local_to_global = {anchor: {p: p for p in range(N_PEOPLE)}}

print(f"{'camera':<8}{'mean sim':>10}{'min sim':>9}   local -> global   (per-pair similarity)")
print(f"{anchor:<8}{'1.000':>10}{'1.000':>9}   identity (anchor)")

for camera in CAMERAS:
    if camera == anchor:
        continue
    G, counts = galleries[camera]

    # Only match identities that actually have detections behind them. An empty local
    # cluster has an all-zero gallery vector, and letting Hungarian pair it up would
    # burn a global slot on nothing -- that is how a camera ends up with an empty track.
    valid_l = [l for l in range(N_PEOPLE) if counts[l] > 0]
    valid_g = [g for g in range(N_PEOPLE) if global_support[g] > 0]
    if not valid_l or not valid_g:
        print(f"{camera:<8}{'--':>10}{'--':>9}   no populated identities to match")
        local_to_global[camera] = {}
        continue

    S_sub = global_gallery[np.ix_(valid_g)] @ G[np.ix_(valid_l)].T
    gi, li = linear_sum_assignment(-S_sub)

    mapping, sims = {}, []
    for gr, lc in zip(gi, li):
        g, l = valid_g[gr], valid_l[lc]
        mapping[l] = g
        sims.append(float(S_sub[gr, lc]))

    # Any local identity left over (more populated locals than populated globals) goes
    # to a global slot nobody is using yet, rather than being silently dropped.
    spare_g = [g for g in range(N_PEOPLE) if g not in mapping.values()]
    for l in valid_l:
        if l not in mapping and spare_g:
            mapping[l] = spare_g.pop(0)
            print(f"  {camera}: local {l} had no global match — parked in "
                  f"person_{mapping[l]} (review this one)")
    local_to_global[camera] = mapping
    S = global_gallery @ G.T          # kept for the printout below

    # Update the running global gallery, weighted by how much evidence backs each.
    for l, g in mapping.items():
        if counts[l] == 0:
            continue
        w_old, w_new = global_support[g], float(counts[l])
        blended = (global_gallery[g] * w_old + G[l] * w_new) / (w_old + w_new)
        global_gallery[g] = blended / (np.linalg.norm(blended) + 1e-12)
        global_support[g] += w_new

    pairs = "  ".join(f"{l}->{g}({S[g, l]:.2f})" for l, g in sorted(mapping.items()))
    print(f"{camera:<8}{np.mean(sims):>10.3f}{np.min(sims):>9.3f}   {pairs}")
    if len(mapping) < N_PEOPLE:
        print(f"{'':8}{'':>19}   only {len(mapping)}/{N_PEOPLE} identities populated "
              f"in this camera")

print("\nHow to sanity-check this table:")
print("  * Mean similarity well above the 'different cams, same frame' median from STEP 8")
print("    means the galleries are genuinely matching.")
print("  * A LOW 'min sim' is the warning sign: one of the six pairings is weakly supported,")
print("    and because Hungarian forces a 1-to-1 assignment it had to pick something. That")
print("    person is the first place to look when proofreading.")

# ── Resolve every detection to a global person id ────────────
cam_global_id = {}
for camera in CAMERAS:
    lid = cam_local_id[camera]
    m = local_to_global[camera]
    gid = np.full(len(lid), -1, dtype=int)
    for i, l in enumerate(lid):
        if l >= 0:
            gid[i] = m.get(int(l), -1)
    cam_global_id[camera] = gid

print(f"\n{'camera':<8}{'assigned':>10}{'unassigned':>12}   detections per global person")
for camera in CAMERAS:
    gid = cam_global_id[camera]
    per = [int((gid == p).sum()) for p in range(N_PEOPLE)]
    print(f"{camera:<8}{int((gid >= 0).sum()):>10}{int((gid < 0).sum()):>12}   {per}")

---

## Part 8: Writing the Identities Back to SLEAP with `sleap-io`

We now have, for every detection, a global person ID. Time to turn that into a `.slp` file
you can open in the SLEAP GUI and proofread.

### The one concept that matters: a `Track` is an object, not a label

This is the single most common mistake when writing tracked `.slp` files:

```python
# WRONG — a new Track object every frame. SLEAP sees thousands of
# one-frame tracks that happen to share a name.
for frame in frames:
    inst.track = sio.Track(name="person_0")

# RIGHT — create the Track objects ONCE, reuse the same object.
tracks = [sio.Track(name=f"person_{k}") for k in range(N_PEOPLE)]
for frame in frames:
    inst.track = tracks[person_id]
```

Identity in SLEAP is **object identity**, not string equality. The same `Track` instance
appearing across many `LabeledFrame`s is what makes it one track.

Because we reuse the same *names* (`person_0` … `person_5`) across all six camera files, and
Stage 3 made those names globally consistent, `person_2` in `reid_CAM1.slp` and `person_2` in
`reid_CAM4.slp` are the same child. That is the property Tutorial 3 needs.

### `Instance` vs `PredictedInstance`

We write **`PredictedInstance`**, not `Instance`:

| | `Instance` | `PredictedInstance` |
|---|---|---|
| Means | a human placed this | a model produced this |
| Has scores | no | yes (per-point + instance) |
| In the GUI | shown as ground truth | shown as a prediction, **and the track-editing tools apply** |

The SLEAP proofreading workflow — "swap tracks from here on", "assign to a new track" — is
built around predicted instances. Writing plain `Instance` objects makes the file look like
finished ground truth and gets in your way when correcting it.

### Unassigned instances keep `track=None`

Detections that Stage 2 dropped (short tracklets, or a 7th person the `N_PEOPLE=6` cap
excluded) are **still written**, just without a track. In the GUI they appear as instances with
no identity, which is exactly right: they are real detections that the algorithm declined to
name, and they are visible for you to assign by hand rather than silently deleted.

In [ ]:
# ============================================================
# STEP 13: Export tracked .slp files with sleap-io
# ============================================================


def make_predicted_instance(pts, skeleton, point_scores, instance_score, track):
    """Build a PredictedInstance, tolerating sleap-io API differences across versions.

    The signature of `PredictedInstance.from_numpy` changed between sleap-io releases
    (argument order and the `instance_score` / `score` name), so we try the known forms
    and fall back to a plain Instance if none apply.
    """
    attempts = (
        lambda: sio.PredictedInstance.from_numpy(
            points=pts, point_scores=point_scores,
            instance_score=instance_score, skeleton=skeleton),
        lambda: sio.PredictedInstance.from_numpy(
            pts, skeleton, point_scores=point_scores, score=instance_score),
        lambda: sio.PredictedInstance.from_numpy(
            points_data=pts, skeleton=skeleton,
            point_scores=point_scores, score=instance_score),
    )
    for attempt in attempts:
        try:
            inst = attempt()
        except TypeError:
            continue
        inst.track = track
        return inst
    inst = sio.Instance.from_numpy(pts, skeleton)   # last resort
    inst.track = track
    return inst


def export_camera(camera):
    """Write reid_results/reid_<CAM>.slp with ReID identities as tracks."""
    src = sio.load_slp(str(POSE_PATHS[camera]))
    skeleton = src.skeletons[0]

    # Point the .slp at the video path valid on THIS machine.
    video = sio.Video(filename=str(VIDEO_PATHS[camera]))

    # Create the Track objects ONCE and reuse them (see the markdown above).
    tracks = [sio.Track(name=f"person_{k}") for k in range(N_PEOPLE)]

    # (frame_idx, instance_idx) -> global person id
    gid_lookup = {
        (int(f), int(i)): int(g)
        for (f, i), g in zip(cam_keys[camera], cam_global_id[camera])
        if g >= 0
    }

    out = sio.Labels(videos=[video], skeletons=[skeleton], tracks=tracks)
    n_tracked = n_untracked = 0

    for lf in sorted(src, key=lambda x: x.frame_idx):
        new_lf = sio.LabeledFrame(video=video, frame_idx=lf.frame_idx)
        for inst_idx, inst in enumerate(lf.instances):
            pts = inst.numpy()
            gid = gid_lookup.get((int(lf.frame_idx), inst_idx))
            track = tracks[gid] if gid is not None else None

            # Reuse the detector's scores when present, otherwise mark visible points 1.0.
            existing = getattr(inst, "points", None)
            scores = None
            if existing is not None:
                try:
                    scores = np.array([getattr(p, "score", np.nan) for p in existing],
                                      dtype=float)
                    if np.isnan(scores).all():
                        scores = None
                except TypeError:
                    scores = None
            if scores is None:
                scores = (~np.isnan(pts[:, 0])).astype(float)

            new_lf.instances.append(
                make_predicted_instance(pts, skeleton, scores, float(np.nanmean(scores)), track)
            )
            if track is None:
                n_untracked += 1
            else:
                n_tracked += 1
        out.append(new_lf)

    path = OUT_DIR / f"reid_{camera}.slp"
    out.save(str(path))
    return path, len(out), n_tracked, n_untracked


print("Exporting tracked .slp files\n")
print(f"{'camera':<8}{'frames':>8}{'tracked':>9}{'untracked':>11}{'MB':>7}   file")
exported = {}
for camera in CAMERAS:
    path, n_frames, n_tr, n_un = export_camera(camera)
    exported[camera] = path
    mb = path.stat().st_size / 1e6
    print(f"{camera:<8}{n_frames:>8}{n_tr:>9}{n_un:>11}{mb:>7.1f}   {path.name}")

print(f"\nWrote {len(exported)} files to {OUT_DIR.resolve()}")

In [ ]:
# ============================================================
# STEP 14: Verify the round-trip — reload and inspect
# ============================================================
# Never trust a write you haven't read back. This catches the "new Track object every
# frame" bug immediately: if that had happened, n_tracks would be in the thousands.

print(f"{'camera':<8}{'tracks':>8}{'frames':>8}{'tracked':>9}{'switches':>10}   "
      f"frames per person")
problems = []
notes = []

for camera in CAMERAS:
    L = sio.load_slp(str(exported[camera]))
    names = [t.name for t in L.tracks]

    per_person = {n: 0 for n in names}
    # An "ID switch" here = a track that vanishes and later reappears. Some of this is
    # legitimate (person leaves the room) but a high count means fragmentation.
    seen_frames = {n: [] for n in names}
    n_tracked = 0
    for lf in L:
        for inst in lf.instances:
            if inst.track is not None:
                per_person[inst.track.name] += 1
                seen_frames[inst.track.name].append(lf.frame_idx)
                n_tracked += 1

    switches = 0
    for n in names:
        fr = np.array(sorted(seen_frames[n]))
        if len(fr) > 1:
            switches += int((np.diff(fr) > 1).sum())

    counts = [per_person[f"person_{k}"] for k in range(N_PEOPLE)]
    print(f"{camera:<8}{len(L.tracks):>8}{len(L):>8}{n_tracked:>9}{switches:>10}   {counts}")

    if len(L.tracks) != N_PEOPLE:
        problems.append(f"{camera}: expected {N_PEOPLE} tracks, found {len(L.tracks)}")
    if names != [f"person_{k}" for k in range(N_PEOPLE)]:
        problems.append(f"{camera}: unexpected track names {names}")
    empty = [n for n, c in per_person.items() if c == 0]
    if empty:
        notes.append(f"{camera}: no instances for {empty}")

print()
if problems:
    print("PROBLEMS (these are real errors):")
    for p in problems:
        print(f"  ! {p}")
else:
    print(f"Structure OK — all {len(CAMERAS)} files have exactly {N_PEOPLE} tracks named "
          f"person_0..person_{N_PEOPLE-1}.")

if notes:
    print("\nEMPTY TRACKS (often legitimate, but worth checking):")
    for n in notes:
        print(f"  - {n}")
    print("  An empty track usually means that person simply never appears in that camera's")
    print("  view, in which case this is the correct answer. It can also mean Stage 2 resolved")
    print("  fewer than N_PEOPLE identities there. Compare against the per-camera identity")
    print("  counts printed in STEP 11 to tell the two apart.")

print("\nCross-camera consistency reminder: person_2 means the same child in every one of")
print("these files. Verify that claim visually in STEP 15 before trusting it downstream —")
print("Stage 3's Hungarian matching cannot tell you when it was forced into a bad pairing.")

In [ ]:
# ============================================================
# STEP 15: Visual check — same person, same colour, all cameras
# ============================================================
# The decisive test of Stage 3: pick one frame, draw every camera side by side with
# each person coloured by their GLOBAL id. If the identities are right, the child in
# red is the same child in all six panels.

CHECK_FRAME = 1000        # try a few different values

# Distinct, colour-blind-friendly palette (RGB 0-255)
PALETTE = [
    (230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48),
    (145, 30, 180), (70, 240, 240), (240, 50, 230), (210, 245, 60),
]

EDGES = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12),
         (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), (0, 5), (0, 6)]


def annotate_frame(camera, frame_idx):
    """Return an RGB frame with global-ID-coloured skeletons drawn on it."""
    cap = cv2.VideoCapture(str(VIDEO_PATHS[camera]))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None

    L = sio.load_slp(str(exported[camera]))
    lf = next((f for f in L if f.frame_idx == frame_idx), None)
    if lf is None:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    for inst in lf.instances:
        pts = inst.numpy()
        if inst.track is None:
            colour_bgr, label = (160, 160, 160), "?"
        else:
            pid = int(inst.track.name.split("_")[1])
            r, g, b = PALETTE[pid % len(PALETTE)]
            colour_bgr, label = (b, g, r), str(pid)

        for a, b_ in EDGES:
            if not (np.isnan(pts[a, 0]) or np.isnan(pts[b_, 0])):
                cv2.line(frame, tuple(pts[a].astype(int)), tuple(pts[b_].astype(int)),
                         colour_bgr, 3)
        for p in pts:
            if not np.isnan(p[0]):
                cv2.circle(frame, tuple(p.astype(int)), 4, colour_bgr, -1)

        vis = pts[~np.isnan(pts[:, 0])]
        if len(vis):
            anchor_pt = vis[np.argmin(vis[:, 1])].astype(int)
            cv2.putText(frame, label, (anchor_pt[0] - 10, anchor_pt[1] - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, colour_bgr, 4)
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(2, 3, figsize=(24, 12))
for ax, camera in zip(axes.flatten(), CAMERAS):
    img = annotate_frame(camera, CHECK_FRAME)
    if img is None:
        ax.text(0.5, 0.5, f"{camera}: could not read frame", ha="center", va="center")
        ax.axis("off")
        continue
    ax.imshow(img)
    ax.set_title(camera, fontsize=13)
    ax.axis("off")

fig.suptitle(f"Frame {CHECK_FRAME} — colour = global person ID  "
             f"(grey '?' = no identity assigned)", fontsize=16, y=0.99)
plt.tight_layout()
plt.savefig(OUT_DIR / f"identity_check_frame{CHECK_FRAME}.png", dpi=120)
plt.show()

print(f"Saved: {OUT_DIR / f'identity_check_frame{CHECK_FRAME}.png'}")
print("\nWhat you are looking for:")
print("  * The same child wears the same colour in every panel where they appear.")
print("  * Within one panel, no colour appears twice (that would be a same-frame collision).")
print("  * Grey '?' skeletons are unassigned — expected for extra people beyond N_PEOPLE.")
print("\nIf colours disagree across cameras, Stage 3's gallery matching mispaired someone;")
print("re-check the 'min sim' column in STEP 12 to find which pairing was weakest.")

---

## Part 9: Proofreading in the SLEAP GUI

The `.slp` files in `reid_results/` are now openable in SLEAP. Automated ReID will not be
perfect, and the point of exporting is to fix it efficiently rather than to pretend otherwise.

### Opening the files

```bash
conda activate sleap
sleap-label reid_results/reid_CAM1.slp
```

If SLEAP cannot find the video, use **Video → Replace Video** and point it at the matching
`aligned_CAM*.mp4`. The `.slp` stores a *path*, not the pixels, so moving between machines
means re-pointing it.

### What to look for, in priority order

1. **Identity swaps.** Two people cross paths and exchange identities. This is the most
   damaging error for triangulation, because it silently mixes two people's 3D skeletons.
   Scrub the timeline watching one colour at a time.
2. **Grey (untracked) instances.** Real detections the algorithm declined to name. Assign
   them, or leave them if they are a person you don't care about.
3. **Fragmentation.** One person split across two tracks with a gap between. Less harmful than
   a swap — triangulation just loses frames — but worth merging.
4. **Cross-camera disagreement.** Hardest to spot in a single-camera GUI, which is why
   STEP 15 renders all six views together. Check that first, then correct in the GUI.

### The keyboard workflow that matters

In SLEAP's **Instances** panel, right-clicking a predicted instance gives you the track tools.
The one you will use most:

- **"Transpose"** / **track swap** — exchange two instances' tracks **from this frame forward**.
  This is the fix for a swap: find the exact frame the identities crossed, swap once, and
  everything after is corrected.

Fixing a swap at its source frame is one action. Correcting it frame by frame is thousands.
Finding the crossing frame precisely is the whole skill.

### Saving your corrections

`File → Save As` to a **new** filename — `reid_CAM1.proofread.slp` — rather than overwriting.
Keeping the algorithm's raw output lets you measure how much correction was needed, which is
the honest way to report how well ReID actually performed.

> **Protect your proofreading.** Hand-corrected identities are the most expensive data in
> this pipeline — they cost human hours and cannot be regenerated. Re-running Tutorial 1
> overwrites `<video>_pose.slp` in place, so any tracks saved under that name are gone.
> This has already happened once on the dataset this tutorial was built from: a set of
> hand-corrected cross-camera identities survived only because they had also been exported
> to an `.analysis.h5` sidecar.
>
> Save proofread output under a distinct name (`*.proofread.slp`), in a directory the
> tutorials never write to, and keep a backup.

In [ ]:
# ============================================================
# STEP 16: Export the cross-camera identity map for Tutorial 3
# ============================================================
# Triangulation needs to know, for each frame and each person, which instance in each
# camera to use. That is exactly what we can now write out.

identity_map = {}           # frame -> person -> {camera: instance_idx}
for camera in CAMERAS:
    for (f, i), g in zip(cam_keys[camera], cam_global_id[camera]):
        if g < 0:
            continue
        identity_map.setdefault(int(f), {}).setdefault(f"person_{int(g)}", {})[camera] = int(i)

map_path = OUT_DIR / "reid_identity_map.json"
with open(map_path, "w") as fh:
    json.dump({"n_people": N_PEOPLE,
               "cameras": CAMERAS,
               "frame_stride": FRAME_STRIDE,
               "anchor_camera": anchor,
               "map": {str(k): v for k, v in sorted(identity_map.items())}},
              fh)
print(f"Wrote {map_path}  ({map_path.stat().st_size/1e6:.1f} MB)")

# ── How usable is this for triangulation? ────────────────────
# A person needs >= 2 cameras in a frame to be triangulated at all, and more cameras
# means a better-conditioned solution.
cam_count_hist = {k: 0 for k in range(len(CAMERAS) + 1)}
per_person_ready = {f"person_{k}": 0 for k in range(N_PEOPLE)}
n_person_frames = 0

for f, people in identity_map.items():
    for person, cams in people.items():
        cam_count_hist[len(cams)] += 1
        n_person_frames += 1
        if len(cams) >= 2:
            per_person_ready[person] += 1

print(f"\nPerson-frames by number of cameras seeing them:")
for k in sorted(cam_count_hist):
    n = cam_count_hist[k]
    if not n:
        continue
    bar = "#" * int(50 * n / max(n_person_frames, 1))
    flag = "  <- cannot triangulate" if k < 2 else ""
    print(f"  {k} cam{'s' if k != 1 else ' '}: {n:>7,} ({n/max(n_person_frames,1):>5.1%}) {bar}{flag}")

triangulable = sum(n for k, n in cam_count_hist.items() if k >= 2)
print(f"\nTriangulable person-frames: {triangulable:,} / {n_person_frames:,} "
      f"({triangulable/max(n_person_frames,1):.1%})")
print(f"\n{'person':<10}{'frames with >=2 cams':>22}")
for p in sorted(per_person_ready):
    print(f"{p:<10}{per_person_ready[p]:>22,}")

print("\nNext: Tutorial 3 reads reid_identity_map.json (or the tracked .slp files directly)")
print("and triangulates each person's 2D keypoints into 3D using calibration.toml.")
print("Proofread first — a single uncorrected identity swap corrupts that person's 3D")
print("trajectory for every frame after it.")

---

## Summary & Next Steps

### What This Notebook Produced

| Output | File | What it is |
|---|---|---|
| Tracked poses per camera | `reid_results/reid_CAM{1..6}.slp` | **The deliverable.** Open in SLEAP to proofread. |
| Embeddings cache | `reid_results/embeddings.npz` | 512-dim vector per detection, so you can re-tune the matching without re-running the encoder |
| Cross-camera identity map | `reid_results/reid_identity_map.json` | `frame -> person -> {camera: instance_idx}` for Tutorial 3 |
| Similarity diagnostics | `reid_results/reid_similarity_analysis.png` | Similarity split by pair type |
| Visual identity check | `reid_results/identity_check_frame*.png` | All six cameras, coloured by global ID |

### The Pipeline, End to End

```
Tutorial 1 output              this notebook                        Tutorial 3
─────────────────              ─────────────                        ──────────
_pose.slp        ──► torso crops ──► OSNet embeddings
(no identities)                            │
                                           ▼
                              Stage 1: per-frame Hungarian
                                (appearance + motion)
                                     -> tracklets
                                           │
                                           ▼
                              Stage 2: merge tracklets,
                              never merging same-frame pairs
                                -> N_PEOPLE per camera
                                           │
                                           ▼
                              Stage 3: Hungarian on galleries
                                -> global person_0..person_5
                                           │
                                           ▼
                              reid_CAM*.slp  ──► PROOFREAD ──► triangulate
```

### The Three Ideas Worth Remembering

1. **Assignment, not thresholding.** "Most similar match" is not a valid identity
   assignment — two people can claim the same partner. The Hungarian algorithm returns a
   genuine one-to-one assignment, and that constraint alone removes a whole class of error.

2. **Hard constraints beat better features.** Two detections in the same frame of the same
   camera are different people. That is free, certain information, and enforcing it in Stage 2
   prevents more mistakes than any amount of embedding tuning.

3. **A `Track` is an object.** Reusing one `sio.Track` instance across frames is what makes
   a track; creating a new one per frame with the same name creates thousands of tracks.

### Known Limitations

- **Stage 3 assumes everyone is visible everywhere.** Hungarian matching on galleries forces a
  1-to-1 pairing, so a person missing from a camera gets mispaired with somebody. Watch the
  `min sim` column in STEP 12 — a low value flags exactly this.
- **`N_PEOPLE` is a hard input, not something discovered.** Set it wrong and Stage 2 either
  merges two people or splits one. The `track=None` instances are the visible symptom.
- **Appearance struggles with matching clothing.** Several children in the same camp T-shirt is
  close to the worst case for a shirt-crop embedding.
- **No geometry is used yet.** The strongest available fix for all three points above is
  epipolar gating from `calibration.toml`: forbid any cross-camera match whose viewing rays
  don't come close to intersecting. Appearance then only has to break the remaining ties.

### Next Tutorial

**Tutorial 3: 3D Triangulation** takes these tracked `.slp` files (proofread first!) plus
`calibration.toml` and fuses each person's 2D keypoints into 3D skeletons with
`sleap-anipose`.

> Proofread before triangulating. One uncorrected identity swap does not degrade the 3D
> result gracefully — it welds two children's bodies together from that frame onward.
